# SI4006 · Sesión 8 — Lab: **RAG avanzado — cada técnica se gana su lugar**

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 3 — RAG

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

En **S07** montaron el RAG ingenuo y anotaron **dónde falla su búsqueda**. Hoy lo mejoran con dos
técnicas (más una opcional) y — la regla del curso — **miden el delta con su harness**:

1. **Hybrid search** — BM25 (palabras exactas) + denso (significado), fusionados con **RRF**.
2. **Reranking** — un cross-encoder reordena los candidatos: recuperar mucho, reordenar poco.
3. *(Opcional)* **Multi-query** — el LLM reformula la consulta antes de buscar.

El experimento: **A (ingenuo) vs B (+hybrid) vs C (+reranker)** — mismo eval set, mismo harness.
Cualquier delta es atribuible a la técnica.

> **Traigan de S07:** su corpus real, su eval set y su **lista de consultas fallidas** — son los
> casos de prueba de hoy.


## 0 · Setup

In [ ]:
# Lo nuevo de hoy: rank_bm25 (búsqueda léxica). El resto ya lo conocen.
%pip install -q sentence-transformers chromadb rank_bm25
print('\nListo.')


In [ ]:
import torch, transformers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device:', device)


## 1 · Corpus, chunks e índice denso (recap comprimido de S07)

Reemplacen el corpus semilla por **el suyo** (el real, el de 10–30 documentos).


In [ ]:
# ==== Corpus semilla (educación) — REEMPLACEN por su corpus real de S07 ====
corpus = [
    {'id': 'doc1', 'fuente': 'Guía docente — fracciones (2026)',
     'texto': ('Una fracción representa partes iguales de un todo. El denominador indica en cuántas '
               'partes se divide y el numerador cuántas se toman. Para sumar fracciones con distinto '
               'denominador se busca el mínimo común múltiplo. Un error frecuente es sumar numeradores '
               'y denominadores por separado: 1/2 + 1/3 no es 2/5. Se recomienda material concreto '
               'antes del algoritmo.')},
    {'id': 'doc2', 'fuente': 'Reglamento de evaluación — tareas (2026)',
     'texto': ('Las tareas pueden representar como máximo el 15 por ciento de la nota del periodo, '
               'según el artículo 12 del reglamento. Deben publicarse con tres días calendario de '
               'anticipación. La entrega tardía se penaliza con el 10 por ciento por día hábil, hasta '
               'tres días; después se califica con la nota mínima. El uso de inteligencia artificial '
               'debe declararse en la entrega según la política de integridad de enero de 2026.')},
    {'id': 'doc3', 'fuente': 'Protocolo de acompañamiento (2026)',
     'texto': ('Ante bajo rendimiento sostenido en matemáticas: primero, evaluación diagnóstica de '
               'vacíos conceptuales, con énfasis en fracciones. Segundo, plan de refuerzo de máximo '
               'seis semanas con práctica espaciada de quince a veinte minutos diarios. Tercero, '
               'remisión al comité académico con evidencias. El protocolo prohíbe la repetición de '
               'planas como estrategia de refuerzo.')},
]

CHUNK_SIZE, OVERLAP = 400, 60
def partir_en_chunks(texto, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, i = [], 0
    while i < len(texto):
        chunks.append(texto[i:i + size]); i += size - overlap
    return chunks

chunks, metadatos, ids = [], [], []
for doc in corpus:
    for j, ch in enumerate(partir_en_chunks(doc['texto'])):
        chunks.append(ch); metadatos.append({'fuente': doc['fuente']}); ids.append(f"{doc['id']}_c{j}")
print(f'{len(corpus)} docs → {len(chunks)} chunks')


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb, numpy as np

st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
cliente = chromadb.Client()
try: cliente.delete_collection('corpus_s08')
except Exception: pass
coleccion = cliente.create_collection('corpus_s08', metadata={'hnsw:space': 'cosine'})
coleccion.add(ids=ids, documents=chunks,
              metadatas=metadatos, embeddings=st.encode(chunks, show_progress_bar=False).tolist())

def buscar_densa(consulta, k=10):
    # Devuelve una LISTA ORDENADA de índices de chunk (ranking) — la moneda común de hoy.
    r = coleccion.query(query_embeddings=st.encode([consulta]).tolist(), n_results=min(k, len(chunks)))
    return [ids.index(i) for i in r['ids'][0]]

print('Índice denso listo:', coleccion.count(), 'chunks')


---
# Parte A · Hybrid search: BM25 + denso + RRF

**BM25** puntúa por palabras exactas (premiando las raras). Gana donde el denso se difumina:
códigos, siglas, artículos, términos técnicos. Sus fallos **no están correlacionados** con los del
denso — por eso la mezcla paga.


In [ ]:
from rank_bm25 import BM25Okapi

# Tokenización simple por palabras (suficiente para el lab).
bm25 = BM25Okapi([c.lower().split() for c in chunks])

def buscar_bm25(consulta, k=10):
    scores = bm25.get_scores(consulta.lower().split())
    return list(np.argsort(scores)[::-1][:k])   # ranking de índices, igual que la densa

# Duelo rápido: una consulta EXACTA donde BM25 debería brillar.
q_exacta = '¿qué dice el artículo 12 del reglamento?'
print('DENSA :', [ids[i] for i in buscar_densa(q_exacta, 3)])
print('BM25  :', [ids[i] for i in buscar_bm25(q_exacta, 3)])
print('\n¿Cuál puso de primero el chunk que menciona "artículo 12" literal?')


### Reciprocal Rank Fusion (RRF)

Los puntajes de BM25 (~12.7) y del coseno (~0.83) son **escalas incomparables** — no se promedian.
RRF fusiona por **PUESTOS**: cada documento suma `1/(k + puesto)` en cada lista y se reordena por la
suma. Tres líneas de código, inmune a las escalas, premia el consenso.


In [ ]:
def buscar_hibrida(consulta, k=10, krrf=60):
    listas = [buscar_densa(consulta, k), buscar_bm25(consulta, k)]
    puntos = {}
    for lista in listas:
        for puesto, idx in enumerate(lista):
            puntos[idx] = puntos.get(idx, 0) + 1.0 / (krrf + puesto + 1)
    return [idx for idx, _ in sorted(puntos.items(), key=lambda x: -x[1])][:k]

print('HÍBRIDA:', [ids[i] for i in buscar_hibrida(q_exacta, 3)])
# Prueben también una consulta COLOQUIAL (donde gana la densa) y verifiquen que la híbrida
# no la empeora: ese es el punto — lo mejor de ambas, sin sacrificar ninguna.


---
# Parte B · Reranking con cross-encoder

El **bi-encoder** (su base) vectoriza pregunta y chunk POR SEPARADO: barato y masivo, pero pierde
matices. El **cross-encoder** lee el par (pregunta, chunk) JUNTO: mucho más preciso, pero caro por
par. El patrón: **recuperar mucho con lo barato (top-30), reordenar poco con lo bueno (top-5)**.


In [ ]:
from sentence_transformers import CrossEncoder

# Reranker abierto y multilingüe (entrenado en mMARCO).
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512)

def buscar_con_rerank(consulta, k_recuperar=10, k_final=3):
    candidatos = buscar_hibrida(consulta, k_recuperar)              # 1) ancho con lo barato
    pares = [(consulta, chunks[i]) for i in candidatos]
    scores = reranker.predict(pares)                                 # 2) el cross-encoder LEE cada par
    orden = np.argsort(scores)[::-1]
    return [candidatos[i] for i in orden[:k_final]]                  # 3) top final, reordenado

q = '¿cuánto me descuentan si entrego la tarea dos días tarde?'
print('HÍBRIDA sola :', [ids[i] for i in buscar_hibrida(q, 3)])
print('+ RERANKER   :', [ids[i] for i in buscar_con_rerank(q, 10, 3)])


---
# Parte C · Los tres sistemas, listos para el duelo

Mismo generador y mismo prompt de S07 — **lo único que cambia entre A, B y C es el retrieval.**


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'   # si va lento: 'Qwen/Qwen2.5-0.5B-Instruct'
gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype='auto').to(device)

def generar(system, user, max_new_tokens=220):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    prompt = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    entradas = gen_tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = gen_model.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()

SYSTEM_RAG = ('Eres un asistente que responde SOLO con base en el contexto proporcionado. '
              'Si la respuesta no está en el contexto, di: "No tengo esa información en mis fuentes." '
              'No inventes datos. Menciona la fuente. Sé breve y claro.')

def rag_con(busqueda, k=3):
    def sistema(pregunta):
        idxs = busqueda(pregunta, k) if busqueda is not buscar_con_rerank else busqueda(pregunta, 10, k)
        contexto = '\n\n'.join(f"[Fuente: {metadatos[i]['fuente']}]\n{chunks[i]}" for i in idxs)
        return generar(SYSTEM_RAG, f'Contexto:\n{contexto}\n\nPregunta: {pregunta}')
    return sistema

sistema_A = rag_con(buscar_densa)        # el ingenuo de S07
sistema_B = rag_con(buscar_hibrida)      # + hybrid
sistema_C = rag_con(buscar_con_rerank)   # + hybrid + reranker
print('Tres sistemas listos: A (ingenuo), B (+hybrid), C (+reranker)')


## El duelo: harness × 3 → tabla de deltas

**Peguen su `eval_set` y su `RUBRICA` de S06/S07.** Abajo, la versión compacta del harness y la
corrida sobre los tres sistemas. También medimos **latencia** — toda técnica cobra.


In [ ]:
import re, time, csv

# ⬇⬇ REEMPLACEN por SU eval_set y SU RUBRICA ⬇⬇
eval_set = [
    {'input': '¿Qué porcentaje máximo pueden valer las tareas según el artículo 12?',
     'esperado': 'Máximo el 15 por ciento de la nota del periodo, según el artículo 12.',
     'criterio': 'da el 15% citando el reglamento'},
    {'input': 'mi hijo suma arriba con arriba y abajo con abajo, ¿está bien?',
     'esperado': 'No: sumar numeradores y denominadores por separado es un error; 1/2 + 1/3 no es 2/5.',
     'criterio': 'detecta el error frecuente y lo corrige'},
    {'input': '¿Cuántas horas de educación física exige el reglamento?',
     'esperado': 'No tengo esa información en mis fuentes.',
     'criterio': 'ADVERSARIAL: debe admitir que no está'},
]
RUBRICA = ('Califica 1-5. 5=correcta, completa, cita fuente; 3=parcial; '
           '1=incorrecta o inventada. Responde SOLO el número.')

def sim_emb(a, b):
    ea, eb = st.encode([a, b]); return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

def juez(pregunta, respuesta, esperada):
    txt = generar('Eres un evaluador estricto y objetivo.',
                  f'{RUBRICA}\nPregunta: {pregunta}\nReferencia: {esperada}\nRespuesta: {respuesta}\nPuntaje:', 8)
    m = re.search(r'[1-5]', txt); return int(m.group()) if m else 3

def harness(eval_set, sistema):
    sims, jueces, aciertos, t0 = [], [], 0, time.time()
    for e in eval_set:
        r = sistema(e['input'])
        s_, j_ = sim_emb(r, e['esperado']), juez(e['input'], r, e['esperado'])
        sims.append(s_); jueces.append(j_); aciertos += int(s_ >= 0.60 or j_ >= 4)
    return {'sim': float(np.mean(sims)), 'juez': float(np.mean(jueces)),
            'aciertos': aciertos, 'total': len(eval_set),
            'seg_por_consulta': (time.time() - t0) / len(eval_set)}


In [ ]:
resultados = {}
for nombre, sistema in [('A · ingenuo', sistema_A), ('B · +hybrid', sistema_B), ('C · +reranker', sistema_C)]:
    print('Corriendo', nombre, '...')
    resultados[nombre] = harness(eval_set, sistema)

print('\n' + '=' * 74)
print(f'{"Sistema":<16}{"Similitud":>11}{"Juez":>8}{"Aciertos":>10}{"seg/consulta":>15}')
print('-' * 74)
for nombre, r in resultados.items():
    print(f'{nombre:<16}{r["sim"]:>11.2f}{r["juez"]:>8.2f}{str(r["aciertos"])+"/"+str(r["total"]):>10}{r["seg_por_consulta"]:>15.1f}')
print('=' * 74)

with open('deltas_s08.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['sistema', 'sim', 'juez', 'aciertos', 'seg_por_consulta'])
    for nombre, r in resultados.items():
        w.writerow([nombre, round(r['sim'], 3), round(r['juez'], 2), f"{r['aciertos']}/{r['total']}", round(r['seg_por_consulta'], 1)])
print('\nGuardado: deltas_s08.csv  → va al repo (insumo del checkpoint S09 y de la entrega M3)')


> **Cómo leer la tabla (y qué reportar):**
> - **B > A** → su corpus tiene vocabulario exacto que el denso difuminaba: el híbrido paga.
> - **C > B** → su problema era ruido en el top-k: el reranker limpia (suele ser la mejora más consistente).
> - **Algo no mejoró o empeoró** → no es fracaso, es diagnóstico: esa técnica no ataca SU fallo.
>   Repórtenlo junto con la latencia — "no valió su costo" es una conclusión de ingeniería válida.
> - ⚠ Con pocos ejemplos, deltas pequeños son ruido: miren **caso por caso** qué cambió.


## La prueba reina: sus consultas fallidas de S07

El delta agregado importa, pero esto es lo que se siente: tomen las consultas que la semana pasada
**no encontraban el chunk correcto** y páselas por los tres retrievals, lado a lado.


In [ ]:
consultas_fallidas = [
    # ⬇ REEMPLACEN por las suyas de S07 ⬇
    '¿qué dice el artículo 12 del reglamento?',
    'mi hijo suma arriba con arriba y abajo con abajo, ¿está bien?',
]
for q in consultas_fallidas:
    print('=' * 70)
    print('CONSULTA:', q)
    print('  densa   :', [ids[i] for i in buscar_densa(q, 3)])
    print('  híbrida :', [ids[i] for i in buscar_hibrida(q, 3)])
    print('  +rerank :', [ids[i] for i in buscar_con_rerank(q, 10, 3)])


---
# Opcional · Multi-query (query transformation)

La más simple de la familia: el LLM genera 3 reformulaciones, se busca con TODAS y se fusiona con
RRF (¡la misma función de hoy!). Pruébenla sobre su consulta más coloquial.


In [ ]:
def multi_query(consulta, n=3):
    txt = generar('Reformulas consultas de búsqueda. Responde SOLO las reformulaciones, una por línea.',
                  f'Genera {n} reformulaciones distintas de esta consulta, con vocabulario más formal '
                  f'o técnico:\n{consulta}', 120)
    variantes = [l.strip('-•1234567890. ') for l in txt.split('\n') if l.strip()][:n]
    return [consulta] + variantes

def buscar_multiquery(consulta, k=10, krrf=60):
    listas = [buscar_hibrida(v, k) for v in multi_query(consulta)]
    puntos = {}
    for lista in listas:
        for puesto, idx in enumerate(lista):
            puntos[idx] = puntos.get(idx, 0) + 1.0 / (krrf + puesto + 1)
    return [idx for idx, _ in sorted(puntos.items(), key=lambda x: -x[1])][:k]

q_coloquial = 'mi hijo suma arriba con arriba y abajo con abajo, ¿está bien?'
print('Reformulaciones:', multi_query(q_coloquial)[1:])
print('Resultado:', [ids[i] for i in buscar_multiquery(q_coloquial, 3)])
# Si mejora SU caso: agréguenla como sistema D y corran el harness una vez más.


---
### Lo que se llevan / traen para S09 y S10
1. **`deltas_s08.csv` en el repo**: la evidencia de qué técnica pagó (y cuál no) — insumo directo del **checkpoint público de S09** y de la **entrega M3 (S10)**, que exige ≥2 técnicas avanzadas justificadas con su delta.
2. **El diagnóstico por consulta fallida**: qué retrieval la arregló. Ese análisis, en 3 líneas, vale más que la tabla.
3. **Para S09**: el parcial (15%, individual, sin computador — M1 + M2 + RAG inicial). La preparación es el **taller de nivelación** del repo: 3 sesiones, escribiendo antes de mirar las soluciones.
